In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, LongType
from pyspark.sql.functions import from_json, col, avg, count, desc, asc, lower, trim, round

spark = (SparkSession.builder.appName("Lab01_Exercise")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.kafka:kafka-clients:3.6.0,org.apache.spark:spark-streaming-kafka-0-10_2.13:4.1.1")
    .master("local[*]")
    .getOrCreate())

# 2. Định nghĩa Schema cho dữ liệu JSON
rating_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True)
])

movie_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

tag_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), True)
])

# 3. Hàm dùng chung để đọc Kafka và parse JSON
def read_and_parse_kafka(topic_name, schema):
    raw_df = spark.read.format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", topic_name) \
        .option("startingOffsets", "earliest") \
        .load()
    
    parsed_df = raw_df.selectExpr("CAST(value AS STRING)") \
        .select(from_json(col("value"), schema).alias("data")) \
        .select("data.*")
    return parsed_df

print("✅ Đã setup xong Spark và hàm đọc Kafka!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/09 22:32:59 WARN Utils: Your hostname, DESKTOP-GEA4IAO, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/09 22:32:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/e/Workspace/bk/year_3/hk252/big_data/lab/.venv_wsl/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/huynhnhat/.ivy2.5.2/cache
The jars for the packages stored in: /home/huynhnhat/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
org.apache.spark#spark-streaming-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c00f7759-8eda-483b-8e4f-e24f5e71a705;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-1

✅ Đã setup xong Spark và hàm đọc Kafka!


In [7]:
ratings_df = read_and_parse_kafka("ratings", rating_schema)
movies_df = read_and_parse_kafka("movies", movie_schema)
tags_df = read_and_parse_kafka("tags", tag_schema)

print("Đã lấy thành công 3 bảng: ratings, movies, tags")

# Xuất dữ liệu
print("Đang xuất mẫu dữ liệu Q1 ra JSON...")

ratings_df.limit(10).coalesce(1).write.mode("overwrite").json("output_exercise/q1_ratings_sample")
movies_df.limit(10).coalesce(1).write.mode("overwrite").json("output_exercise/q1_movies_sample")
tags_df.limit(10).coalesce(1).write.mode("overwrite").json("output_exercise/q1_tags_sample")

print("✅ Đã xuất thành công 3 thư mục mẫu Q1!")

Đã lấy thành công 3 bảng: ratings, movies, tags
Đang xuất mẫu dữ liệu Q1 ra JSON...
✅ Đã xuất thành công 3 thư mục mẫu Q1!


In [8]:
# Tính điểm trung bình và đếm số lượng đánh giá cho mỗi phim
movie_stats = ratings_df.groupBy("movieId").agg(
    count("*").alias("total_ratings"),
    round(avg("rating"), 4).alias("avg_rating")
)

# Lọc các phim có > 30 lượt đánh giá, join để lấy tên phim và lấy top 5
top_5_movies = movie_stats.filter(col("total_ratings") > 30) \
    .join(movies_df, on="movieId", how="inner") \
    .select("movieId", "title", "avg_rating", "total_ratings") \
    .orderBy(desc("avg_rating"), desc("total_ratings")) \
    .limit(5)

print("--- TOP 5 MOVIES (Hơn 30 lượt đánh giá) ---")
top_5_movies.show(truncate=False)
# Tạo thư mục và xuất kết quả ra file JSON
print("Đang xuất Top 5 Movies ra thư mục dạng JSON...")
top_5_movies.coalesce(1).write \
    .mode("overwrite") \
    .json("output_exercise/q2_top5_movies_json")

print("✅ Đã xuất thành công tại: output_exercise/q2_top5_movies_json")

--- TOP 5 MOVIES (Hơn 30 lượt đánh giá) ---


+-------+--------------------------------+----------+-------------+
|movieId|title                           |avg_rating|total_ratings|
+-------+--------------------------------+----------+-------------+
|318    |Shawshank Redemption, The (1994)|4.429     |317          |
|1204   |Lawrence of Arabia (1962)       |4.3       |45           |
|858    |Godfather, The (1972)           |4.2891    |192          |
|2959   |Fight Club (1999)               |4.2729    |218          |
|1276   |Cool Hand Luke (1967)           |4.2719    |57           |
+-------+--------------------------------+----------+-------------+

Đang xuất Top 5 Movies ra thư mục dạng JSON...


✅ Đã xuất thành công tại: output_exercise/q2_top5_movies_json


In [9]:
# Làm sạch cột tag (đưa về chữ thường, xóa khoảng trắng thừa) để tránh lỗi tag rác
cleaned_tags = tags_df.select(
    "movieId",
    lower(trim(col("tag"))).alias("clean_tag")
).filter(col("clean_tag").isNotNull() & (col("clean_tag") != "")).dropDuplicates(["movieId", "clean_tag"])

# Join với bảng movie_stats (bảng chứa điểm trung bình của từng phim)
tag_movie_join = cleaned_tags.join(movie_stats, on="movieId", how="inner")

# Nhóm theo tag để xem điểm trung bình của các phim mang tag đó là bao nhiêu
tag_stats = tag_movie_join.groupBy("clean_tag").agg(
    round(avg("avg_rating"), 4).alias("tag_avg_score"),
    count("movieId").alias("movie_count")
).orderBy(asc("tag_avg_score"))

worst_5_tags = tag_stats.limit(5)

print("--- TOP 5 WORST TAGS ---")
worst_5_tags.show(truncate=False)
# Tạo thư mục và xuất kết quả ra file JSON
print("Đang xuất 5 Worst Tags ra thư mục dạng JSON...")
worst_5_tags.coalesce(1).write \
    .mode("overwrite") \
    .json("output_exercise/q3_worst_tags_json")

print("✅ Đã xuất thành công tại: output_exercise/q3_worst_tags_json")

--- TOP 5 WORST TAGS ---


+---------+-------------+-----------+
|clean_tag|tag_avg_score|movie_count|
+---------+-------------+-----------+
|symbolic |0.5          |1          |
|stage    |1.75         |1          |
|tokyo    |2.0          |1          |
|snl      |2.1          |1          |
|jungle   |2.1364       |1          |
+---------+-------------+-----------+

Đang xuất 5 Worst Tags ra thư mục dạng JSON...


✅ Đã xuất thành công tại: output_exercise/q3_worst_tags_json


In [11]:
# Tính điểm trung bình toàn cục (của tất cả các phim)
global_avg_row = movie_stats.agg(round(avg("avg_rating"), 4).alias("global_avg")).collect()[0]
global_avg_val = global_avg_row["global_avg"]

# Tính điểm trung bình của riêng nhóm phim bị dính 5 worst tags
worst_group_avg_row = worst_5_tags.agg(round(avg("tag_avg_score"), 4).alias("worst_group_avg")).collect()[0]
worst_group_avg_val = worst_group_avg_row["worst_group_avg"]

print(f"-> Điểm đánh giá trung bình TOÀN BỘ phim: {global_avg_val}")
print(f"-> Điểm đánh giá trung bình của nhóm WORST TAGS: {worst_group_avg_val}")
print("-" * 50)

# In kết luận tự động
if worst_group_avg_val < global_avg_val:
    print("KẾT LUẬN QUAN SÁT:")
    print("Các bộ phim gắn những tag này có xu hướng nhận điểm đánh giá thấp hơn mặt bằng chung của hệ thống.")
    print("Tuy nhiên, lưu ý đây chỉ là sự tương quan (correlation), không đủ để kết luận đây là nguyên nhân (causation).")
else:
    print("KẾT LUẬN QUAN SÁT:")
    print("Nhóm tag này không ảnh hưởng tiêu cực đến điểm đánh giá trung bình chung.")

# Tìm danh sách các phim cụ thể mang 5 worst tags
q4_movies = worst_5_tags.select("clean_tag") \
    .join(cleaned_tags, on="clean_tag", how="inner") \
    .join(movies_df, on="movieId", how="inner") \
    .select("clean_tag", "movieId", "title")

# Hiển thị thử ra màn hình cho vui
print("\n--- DANH SÁCH PHIM THUỘC 5 WORST TAGS ---")
q4_movies.show(truncate=False)

# Xuất kết quả ra thư mục JSON
print("Đang xuất danh sách phim Q4 ra JSON...")
q4_movies.coalesce(1).write \
    .mode("overwrite") \
    .json("output_exercise/q4_movies_per_worst_tag")

print("✅ Đã xuất thành công tại: output_exercise/q4_movies_per_worst_tag")

-> Điểm đánh giá trung bình TOÀN BỘ phim: 3.2624
-> Điểm đánh giá trung bình của nhóm WORST TAGS: 1.6973
--------------------------------------------------
KẾT LUẬN QUAN SÁT:
Các bộ phim gắn những tag này có xu hướng nhận điểm đánh giá thấp hơn mặt bằng chung của hệ thống.
Tuy nhiên, lưu ý đây chỉ là sự tương quan (correlation), không đủ để kết luận đây là nguyên nhân (causation).

--- DANH SÁCH PHIM THUỘC 5 WORST TAGS ---


+---------+-------+---------------------------------------------+
|clean_tag|movieId|title                                        |
+---------+-------+---------------------------------------------+
|stage    |8943   |Being Julia (2004)                           |
|tokyo    |6407   |Walk, Don't Run (1966)                       |
|jungle   |1474   |Jungle2Jungle (a.k.a. Jungle 2 Jungle) (1997)|
|symbolic |26717  |Begotten (1990)                              |
|snl      |2296   |Night at the Roxbury, A (1998)               |
+---------+-------+---------------------------------------------+

Đang xuất danh sách phim Q4 ra JSON...


✅ Đã xuất thành công tại: output_exercise/q4_movies_per_worst_tag
